# 05. Kiểm chứng Độc lập Toàn diện trên `PEMS04` và Kết luận Toàn Dự án

> Notebook cuối cùng của lộ trình `5` bước: lặp lại **đầy đủ và đối xứng `100%` thí nghiệm đối chứng `6` cấu hình đặc trưng** (`Raw 3 → Base 7 → +Nhóm 1 / +Nhóm 2 / +Nhóm 3 → Full 14`) trên tập bản đồ độc lập **`PEMS04`** (`307` trạm, khu vực Vịnh San Francisco — chưa từng chạm tới trong quá trình khám phá `NB03` và lựa chọn đặc trưng `NB04`). Toàn bộ so sánh giữ **công bằng tuyệt đối (*ceteris paribus*)** trên cùng một kiến trúc cố định **`ST-GAT (LSTM → 2-layer Multi-Head GATv2)`**, cùng tiền xử lý **Chen et al. (2003)**, cùng chuẩn chia **`60 / 20 / 20` (`12 → 12`)** (**LibCity, 2021**; **BasicTS, 2025**) và cùng hạt giống `SEED = 42`.

### Tóm tắt
* **Vì sao không so chéo với số liệu từ bài báo khác (`DCRNN / STGCN / STID`):** Bài báo gốc **ST-GAT (Zhang et al., 2019)** chỉ đánh giá trên `PeMSD7`, còn các benchmark bên ngoài huấn luyện trên dữ liệu thô chưa làm sạch và chỉ dùng `1` biến `Flow`. Do đó, phép kiểm chứng công bằng và chặt chẽ nhất cho giả thuyết đặc trưng (`RQ2`) là **đối chứng trực tiếp với chính `ST-GAT (LSTM → GATv2)` gốc (`Raw 3` và `Base 7`) trên cả hai thành phố độc lập `PEMS08` và `PEMS04`**.
* **Cấu trúc 5 phần:**
  1. **Phần 1:** Kiểm toán `PEMS04` (`60 / 20 / 20`) và tự hiệu chuẩn ngưỡng tắc nghẽn thuần `Train (60%)` đối chiếu với `PEMS08`.
  2. **Phần 2:** Đối chứng toàn bộ **`6` cấu hình đặc trưng** trên `PEMS04` (huấn luyện độc lập trên `PEMS04` kèm kiểm chứng chuyển giao *Zero-Shot* từ `NB04` ở `0` epoch).
  3. **Phần 3:** Kiểm chứng tính nhất quán xuyên thành phố của **`Permutation Feature Importance`** (`PEMS08` vs. `PEMS04`) và trọng số chú ý cạnh động **`GATv2` ($\alpha_{ij}$)**.
  4. **Phần 4:** Đối chiếu tổng hợp `PEMS08` vs. `PEMS04`, độ nhạy tầm dự báo (`5p → 60p`) và đánh giá phụ đánh dấu điểm tắc nghẽn.
  5. **Phần 5:** **Kết luận Toàn bộ Dự án (`RQ1 + RQ2`)**, đóng góp chính cho bài báo IEEE và giới hạn nghiên cứu.

---
## 1. Kiểm toán `PEMS04` (`60 / 20 / 20`) và Tự hiệu chuẩn Ngưỡng Thuần `Train`

Câu hỏi mở đầu kiểm chứng chéo: *Khi đưa công thức tự tính ngưỡng từ Biểu đồ cơ bản (`RQ1`) sang bản đồ độc lập `PEMS04` (`307` trạm, đồ thị thưa hơn và mật độ nền khác `PEMS08`), các ngưỡng $v_{\text{cong}}, o_{\text{sat}}$ hiệu chuẩn thuần trên `Train (60%)` có tự thích nghi hợp lý mà không cần chỉnh tay hằng số hay không?*

In [ ]:
import os
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
from scipy.stats import spearmanr
import torch

from src.data import FEATURE_CONFIGS, prepare_pems_bundle
from src.explain import compute_permutation_importance, extract_gatv2_edge_attention
from src.models import STGATv2
from src.plots_nb05 import (
    apply_paper_style,
    plot_cross_map_pfi_and_attention,
    plot_final_cross_map_synthesis,
    plot_pems04_ablation_mae,
    plot_pems04_vs_pems08_thresholds,
)
from src.train import (
    evaluate_multistep_forecast,
    predict_anchors,
    set_seed,
    train_stgatv2_experiment,
)

SEED = 42
set_seed(SEED)
torch.set_num_threads(min(8, os.cpu_count() or 4))
apply_paper_style()

CLEAN_DIR = ROOT / "datasets" / "cleaned"
RAW_DIR = ROOT / "datasets" / "data"
OUT_DIR = ROOT / "outputs"
MODEL_DIR = OUT_DIR / "models"
FIG_DIR = OUT_DIR / "figures"

bundle08 = prepare_pems_bundle("PEMS08", clean_dir=CLEAN_DIR, raw_dir=RAW_DIR)
bundle04 = prepare_pems_bundle("PEMS04", clean_dir=CLEAN_DIR, raw_dir=RAW_DIR)

th08 = bundle08["train_thresholds"]
th04 = bundle04["train_thresholds"]
em04 = bundle04["edge_meta"]
cg04 = bundle04["is_congested"]
t_tr4, t_va4 = bundle04["t_train_end"], bundle04["t_val_end"]
T4, N4, _ = bundle04["data_raw"].shape

print(f"PEMS04: {T4:,} bước × {N4} trạm | Đồ thị 2 chiều: {em04['n_edges_total']} cạnh ({em04['n_self']} Self + {em04['n_in']} In + {em04['n_out']} Out)")
print(f"Trạm điểm nghẽn thuần Train PEMS04 (>= 1%): {len(bundle04['bottleneck_nodes'])} / {N4} trạm (Top 5: {bundle04['bottleneck_nodes'][:5].tolist()})")

th_compare_df = pd.DataFrame(
    [
        {
            "Bản đồ (hiệu chuẩn thuần Train 60%)": "PEMS08 (San Bernardino — 170 trạm)",
            "v_ff (mph)": round(th08.v_ff, 2),
            "v_cong = 0,60 v_ff (mph)": round(th08.v_cong, 2),
            "o_crit": round(th08.o_crit, 4),
            "o_sat (90% Q_max nhánh lên)": round(th08.o_sat, 4),
            "Tắc Train / Val / Test (%)": f"{bundle08['is_congested'][:bundle08['t_train_end']].mean()*100:.2f} / {bundle08['is_congested'][bundle08['t_train_end']:bundle08['t_val_end']].mean()*100:.2f} / {bundle08['is_congested'][bundle08['t_val_end']:].mean()*100:.2f}",
        },
        {
            "Bản đồ (hiệu chuẩn thuần Train 60%)": "PEMS04 (San Francisco Bay — 307 trạm)",
            "v_ff (mph)": round(th04.v_ff, 2),
            "v_cong = 0,60 v_ff (mph)": round(th04.v_cong, 2),
            "o_crit": round(th04.o_crit, 4),
            "o_sat (90% Q_max nhánh lên)": round(th04.o_sat, 4),
            "Tắc Train / Val / Test (%)": f"{cg04[:t_tr4].mean()*100:.2f} / {cg04[t_tr4:t_va4].mean()*100:.2f} / {cg04[t_va4:].mean()*100:.2f}",
        },
    ]
)
display(th_compare_df)

split04_df = pd.DataFrame(
    [
        {"Phân đoạn PEMS04 (60/20/20)": "Train (60%)", "Chỉ số thời gian": f"[0, {t_tr4 - 1}]", "Số ngày (~)": round(t_tr4 / 288.0, 2), "Số mẫu cửa sổ (12→12)": len(bundle04["t_train"]), "Tỷ lệ Tắc thực (%)": round(float(cg04[:t_tr4].mean() * 100.0), 2)},
        {"Phân đoạn PEMS04 (60/20/20)": "Validation (20%)", "Chỉ số thời gian": f"[{t_tr4}, {t_va4 - 1}]", "Số ngày (~)": round((t_va4 - t_tr4) / 288.0, 2), "Số mẫu cửa sổ (12→12)": len(bundle04["t_val"]), "Tỷ lệ Tắc thực (%)": round(float(cg04[t_tr4:t_va4].mean() * 100.0), 2)},
        {"Phân đoạn PEMS04 (60/20/20)": "Test (20%)", "Chỉ số thời gian": f"[{t_va4}, {T4 - 1}]", "Số ngày (~)": round((T4 - t_va4) / 288.0, 2), "Số mẫu cửa sổ (12→12)": len(bundle04["t_test"]), "Tỷ lệ Tắc thực (%)": round(float(cg04[t_va4:].mean() * 100.0), 2)},
    ]
)
display(split04_df)

plot_pems04_vs_pems08_thresholds(
    bundle08["data_raw"][: bundle08["t_train_end"]],
    th08,
    bundle04["data_raw"][:t_tr4],
    th04,
    FIG_DIR / "nb05_pems04_thresholds_vs_pems08.png",
)

> **Nhận xét (Công thức tự tính ngưỡng RQ1 tự thích nghi chuẩn xác sang `PEMS04`):**
>
> * **Tự động điều chỉnh $o_{\text{sat}}$ theo đặc trưng mật độ từng bản đồ (Hình 1A–B):** Trên `Train (60%)` của `PEMS04`, vận tốc tự do và ngưỡng tắc nghẽn FHWA gần như đồng nhất với `PEMS08` ($v_{\text{ff}} = 67{,}20\text{ mph} \Rightarrow v_{\text{cong}} = 40{,}32\text{ mph}$ so với `40,26 mph`). Ngược lại, vì mật độ mặt đường trung bình của `PEMS04` thấp hơn (`0,053` vs `0,065`), thuật toán tự động hạ ngưỡng bão hòa từ $o_{\text{sat}} = 0{,}1375$ (`PEMS08`) xuống **$o_{\text{sat}} = \mathbf{0{,}1025}$ (`PEMS04`, đỉnh $o_{\text{crit}} = 0{,}1525$)** — chứng minh trực quan lý do không được áp cứng một hằng số `Occ ≥ 0,15` cho mọi bản đồ.
>
> * **Phân bổ tập chia `60 / 20 / 20` ổn định và không rò rỉ:** Với ngưỡng thuần `Train`, tỷ lệ tắc thực trên `PEMS04` ổn định ở mức `2,84%` (`Train`), `4,05%` (`Validation`), `3,12%` (`Test`) với `155 / 307` trạm điểm nghẽn và `10.161 / 3.364 / 3.365` mẫu cửa sổ cách ly biên `22` bước.
>
> **Câu hỏi nghi vấn tiếp theo:**
>
> Khi lặp lại đúng **`6` cấu hình đặc trưng** (`Raw 3 → Base 7 → +Nhóm 1 / +Nhóm 2 / +Nhóm 3 → Full 14`) trên kiến trúc đóng băng **`ST-GAT (LSTM → 2-layer GATv2)`** ở bản đồ độc lập **`PEMS04`** (cả khi huấn luyện mới lẫn khi chuyển giao *Zero-Shot* từ `PEMS08`), mức cải thiện `MAE` có còn tái lập nhất quán như trên `PEMS08` không?

---
## 2. Đối chứng Toàn bộ `6` Cấu hình Đặc trưng trên `PEMS04` (Huấn luyện Độc lập & Kiểm chứng *Zero-Shot*)

Giữ **đóng băng `100%` kiến trúc `ST-GAT (LSTM → 2-layer Multi-Head GATv2)`** và siêu tham số y hệt `NB04` (`hidden_dim = 32`, `4` heads, `L1Loss / MAE`, `batch_size = 128`, `epochs = 20`, `lr = 5e-3`, `SEED = 42`), ta thực hiện hai phép đối chứng hoàn toàn công bằng trên `PEMS04`:
1. **Thí nghiệm chính — Huấn luyện độc lập cả `6` cấu hình trên `PEMS04 Train (60%)`:** Đo trực tiếp `MAE` trên `PEMS04 Test (20%)` (nạp checkpoint đã lưu tại `outputs/models/stgatv2_pems04_*.pt`; đặt `FORCE_RETRAIN_PEMS04 = True` để huấn luyện lại từ đầu).
2. **Thí nghiệm bổ trợ — Chuyển giao quy nạp *Zero-Shot* (`PEMS08 → PEMS04`, `0` epoch):** Nạp thẳng `6` checkpoint của `NB04` sang đồ thị `307` trạm của `PEMS04` để kiểm tra tính bất biến địa lý của từng nhóm đặc trưng.

In [ ]:
FORCE_RETRAIN_PEMS04 = False
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

TAG_MAP = {
    "Raw (3)": "raw3",
    "Base (7)": "base7",
    "Base + Nhóm 1 (10)": "base_g1",
    "Base + Nhóm 2 (9)": "base_g2",
    "Base + Nhóm 3 (9)": "base_g3",
    "Full (14)": "full14",
}

# 1. All 6 controlled configs trained independently on PEMS04
results04 = {}
histories04 = {}
rows04 = []

for cfg_name, feat_idx in FEATURE_CONFIGS.items():
    tag = TAG_MAP[cfg_name]
    res = train_stgatv2_experiment(
        bundle=bundle04,
        config_name=cfg_name,
        feature_indices=feat_idx,
        hidden_dim=32,
        num_lstm_layers=1,
        num_gat_layers=2,
        num_heads=4,
        dropout=0.10,
        batch_size=128,
        epochs=20,
        lr=5e-3,
        weight_decay=1e-4,
        patience=5,
        seed=SEED,
        save_path=MODEL_DIR / f"stgatv2_pems04_{tag}.pt",
        load_if_exists=(not FORCE_RETRAIN_PEMS04),
        verbose=True,
    )
    results04[cfg_name] = res
    histories04[cfg_name] = res["history"]
    ov = res["test_eval"]["overall"]
    rows04.append(
        {
            "Cấu hình": cfg_name,
            "Số kênh": len(feat_idx),
            "Tham số": res["n_params"],
            "Val L1": round(res["best_val_loss"], 4),
            "Std-MAE (3 biến)": round(ov["std_mae"], 4),
            "MAE Speed (mph)": round(ov["mae_speed"], 3),
            "RMSE Speed (mph)": round(ov["rmse_speed"], 3),
            "MAE Occ": round(ov["mae_occ"], 5),
            "MAE Flow (xe/5p)": round(ov["mae_flow"], 2),
            "MAE Speed Điểm nghẽn (mph)": round(ov["mae_speed_bn"], 3),
            "MAE Flow Điểm nghẽn": round(ov["mae_flow_bn"], 2),
        }
    )

ablation04_df = pd.DataFrame(rows04)
raw04_std = float(ablation04_df.loc[ablation04_df["Cấu hình"] == "Raw (3)", "Std-MAE (3 biến)"].iloc[0])
base04_std = float(ablation04_df.loc[ablation04_df["Cấu hình"] == "Base (7)", "Std-MAE (3 biến)"].iloc[0])
ablation04_df["So với Raw (%)"] = [f"{(v - raw04_std) / raw04_std * 100.0:+.2f}%" for v in ablation04_df["Std-MAE (3 biến)"]]
ablation04_df["So với Base (%)"] = [f"{(v - base04_std) / base04_std * 100.0:+.2f}%" for v in ablation04_df["Std-MAE (3 biến)"]]
ablation04_df.to_csv(OUT_DIR / "nb05_pems04_ablation_summary.csv", index=False)

print("Bảng 2.1 — Đối chứng 6 cấu hình đặc trưng huấn luyện độc lập trên PEMS04 (Test Avg 5–60p):")
display(ablation04_df)

# 2. Zero-Shot Transfer check: all 6 NB04 PEMS08 checkpoints evaluated directly on PEMS04 Test (0 epochs)
zs_rows = []
zs_full_eval = None
targets04_t = torch.from_numpy(np.ascontiguousarray(bundle04["targets_scaled"], dtype=np.float32))
for cfg_name, feat_idx in FEATURE_CONFIGS.items():
    tag = TAG_MAP[cfg_name]
    ck = torch.load(MODEL_DIR / f"stgatv2_pems08_{tag}.pt", map_location=device, weights_only=False)
    m_zs = STGATv2(
        len(feat_idx), bundle04["edge_index"], bundle04["edge_type"],
        hidden_dim=ck["hidden_dim"], num_lstm_layers=ck["num_lstm_layers"],
        num_gat_layers=ck["num_gat_layers"], num_heads=ck["num_heads"],
        horizon_steps=ck["horizon_steps"], dropout=ck["dropout"],
    ).to(device)
    m_zs.load_state_dict({k: v for k, v in ck["state_dict"].items() if k not in ("edge_index", "edge_type")}, strict=False)
    m_zs.eval()
    ft = torch.from_numpy(np.ascontiguousarray(bundle04["feats_scaled"][:, :, feat_idx], dtype=np.float32))
    yp_zs, yt_zs = predict_anchors(m_zs, ft, targets04_t, bundle04["t_test"], 12, 12, device, batch_size=256)
    ev_zs = evaluate_multistep_forecast(yt_zs, yp_zs, bundle04["target_mean"], bundle04["target_std"], bundle04["train_thresholds"], bundle04["bottleneck_mask"])
    if cfg_name == "Full (14)":
        zs_full_eval = ev_zs
    ov = ev_zs["overall"]
    zs_rows.append(
        {
            "Checkpoint từ PEMS08 (Zero-Shot, 0 ep)": cfg_name,
            "Std-MAE": round(ov["std_mae"], 4),
            "MAE Speed (mph)": round(ov["mae_speed"], 3),
            "MAE Occ": round(ov["mae_occ"], 5),
            "MAE Flow (xe/5p)": round(ov["mae_flow"], 2),
            "F1 Tắc 60p (phụ)": round(ev_zs["by_horizon"][60]["cong_f1"], 4),
        }
    )

zs_df = pd.DataFrame(zs_rows)
zs_raw = float(zs_df.loc[zs_df["Checkpoint từ PEMS08 (Zero-Shot, 0 ep)"] == "Raw (3)", "Std-MAE"].iloc[0])
zs_base = float(zs_df.loc[zs_df["Checkpoint từ PEMS08 (Zero-Shot, 0 ep)"] == "Base (7)", "Std-MAE"].iloc[0])
zs_df["So với Raw (%)"] = [f"{(v - zs_raw) / zs_raw * 100.0:+.2f}%" for v in zs_df["Std-MAE"]]
zs_df["So với Base (%)"] = [f"{(v - zs_base) / zs_base * 100.0:+.2f}%" for v in zs_df["Std-MAE"]]
zs_df.to_csv(OUT_DIR / "nb05_pems04_zeroshot_summary.csv", index=False)

print("Bảng 2.2 — Kiểm chứng chuyển giao quy nạp Zero-Shot (PEMS08 → PEMS04, 0 epoch):")
display(zs_df)

plot_pems04_ablation_mae(ablation04_df, histories04, FIG_DIR / "nb05_pems04_ablation_mae.png")

> **Nhận xét (Tái lập nhất quán `100%` hiệu quả các nhóm đặc trưng trên bản đồ độc lập `PEMS04`):**
>
> * **Huấn luyện độc lập trên `PEMS04` tái lập chính xác mọi kết luận của `NB04` (Bảng 2.1 & Hình 2):**
>
>   1. **Nhóm 4 `STID` (`Raw 3 → Base 7`):** Giảm mạnh `Val L1` (`0,2084 → 0,1971`) và `Test Std-MAE` (**`-4,94%`**, `0,1904 → 0,1810`; `MAE Flow -7,71%`, `MAE Occ -5,16%`, `MAE Speed -2,81%`).
>
>   2. **Cả `3` nhóm tín hiệu tiền tắc nghẽn chạy riêng lẻ đều vượt `Base (7)` trên cả `Val L1` lẫn `Test Std-MAE`:** **Nhóm 1 Dao động khử dốc tiếp tục đứng #1** (`0,1790`, `-1,10%` vs `Base`), kế tiếp là **Nhóm 3 Độ dốc có dấu** (`0,1795`, `-0,83%`) và **Nhóm 2 Liên kết đa biến** (`0,1803`, `-0,39%`).
>
>   3. **Tổ hợp `Full (14)` tiếp tục đứng số 1 tuyệt đối trên toàn bộ chỉ số hồi quy:** `Val L1 = 0,1942`, **`Test Std-MAE = 0,1785`** (**`-6,25%`** so với `Raw 3`, **`-1,38%`** so với `Base 7`), `MAE Speed = 1,737 mph`, `MAE Occ = 0,00849`, `MAE Flow = 22,39 xe/5p`, và `MAE Speed Điểm nghẽn = 2,295 mph`.
>
> * **Phát hiện bổ trợ sâu sắc từ chuyển giao *Zero-Shot* (`PEMS08 → PEMS04`, `0` epoch — Bảng 2.2):** Dù không huấn luyện lại, cả `4` cấu hình có tín hiệu tiền tắc nghẽn (`+Nhóm 1: 0,1947`, `+Nhóm 2: 0,1945`, `+Nhóm 3: 0,1932`, `Full 14: 0,1939`) đều vượt đồng thời cả `Raw (3)` (`0,1974`) lẫn `Base (7)` (`0,1959`). Đáng chú ý, vì khung giờ cao điểm giữa San Bernardino (mùa hè) và Vịnh San Francisco (mùa đông) lệch pha nhau nên Nhóm 4 `STID` gặp *domain shift* nhẹ ở biến `Speed` (`1,877 → 1,912`), còn `7` tín hiệu tiền tắc nghẽn vật lý **bất biến theo địa lý** nên vẫn kéo giảm đồng thời cả `Speed`, `Occ`, `Flow` so với `Base (7)` ngay ở `0` epoch! Khi huấn luyện thích nghi địa phương trên `PEMS04`, `Full (14)` giảm thêm `-7,94% Std-MAE` (`0,1939 → 0,1785`).
>
> **Câu hỏi nghi vấn tiếp theo:**
>
> Bên trong `ST-GAT Full (14)` huấn luyện trên `PEMS04`, thứ hạng đóng góp của **`4` nhóm đặc trưng** và **`14` biến riêng lẻ** (`Test PFI`), cũng như cơ chế chú ý cạnh động **`GATv2` (`Self / In / Out`)** có nhất quán với `PEMS08` (`NB04`) hay không?

---
## 3. Kiểm chứng Xuyên Thành phố: Đóng góp Đặc trưng (`Test PFI`) và Chú ý `GATv2` trên `PEMS04`

Ta chạy `Permutation Feature Importance` (`5` lần lặp, `1.024` mẫu `Test`, `SEED = 42`) và trích xuất trọng số chú ý cạnh động `GATv2` trên **`ST-GAT Full (14)`** của `PEMS04`, sau đó đối chiếu trực tiếp với kết quả `PEMS08` (`NB04`) bằng hệ số tương quan hạng **Spearman $\rho$**.

In [ ]:
fidx14 = FEATURE_CONFIGS["Full (14)"]
model04_full = results04["Full (14)"]["model"]

pfi04 = compute_permutation_importance(
    model=model04_full,
    bundle=bundle04,
    feature_indices=fidx14,
    n_repeats=5,
    max_samples=1024,
    batch_size=256,
    seed=SEED,
)
grp04_df = pfi04["group_df"]
feat04_df = pfi04["feature_df"]
feat04_df.to_csv(OUT_DIR / "nb05_pems04_pfi.csv", index=False)

# Load PEMS08 Full (14) PFI for exact side-by-side comparison
feat08_df = pd.read_csv(OUT_DIR / "nb04_permutation_importance.csv")
ckpt08_full = torch.load(MODEL_DIR / "stgatv2_pems08_full14.pt", map_location=device, weights_only=False)
model08_full = STGATv2(14, bundle08["edge_index"], bundle08["edge_type"], hidden_dim=32, num_lstm_layers=1, num_gat_layers=2, num_heads=4, horizon_steps=12, dropout=0.10).to(device)
model08_full.load_state_dict(ckpt08_full["state_dict"])
model08_full.eval()
pfi08 = compute_permutation_importance(model08_full, bundle08, fidx14, n_repeats=5, max_samples=1024, batch_size=256, seed=SEED)
grp08_df = pfi08["group_df"]

m_rank = feat08_df[["feature", "group", "delta_std_mae"]].rename(columns={"delta_std_mae": "PEMS08 ΔStd-MAE"}).merge(
    feat04_df[["feature", "delta_std_mae", "delta_mae_speed"]].rename(columns={"delta_std_mae": "PEMS04 ΔStd-MAE", "delta_mae_speed": "PEMS04 ΔSpeed"}),
    on="feature",
)
rho_14, p_14 = spearmanr(m_rank["PEMS08 ΔStd-MAE"], m_rank["PEMS04 ΔStd-MAE"])
m_11 = m_rank[~m_rank["group"].str.startswith("Raw")]
rho_11, p_11 = spearmanr(m_11["PEMS08 ΔStd-MAE"], m_11["PEMS04 ΔStd-MAE"])

print(f"Tương quan hạng Spearman của PFI giữa PEMS08 và PEMS04: đủ 14 biến ρ = {rho_14:.4f} (p = {p_14:.2e}) | riêng 11 biến bổ sung ρ = {rho_11:.4f} (p = {p_11:.2e})")
print(f"Số đặc trưng có đóng góp dương (ΔStd-MAE > 0) trên PEMS04: {int((feat04_df['delta_std_mae'] > 0).sum())} / 14")

grp_cmp_df = grp08_df[["group", "n_features", "delta_std_mae"]].rename(columns={"group": "Nhóm đặc trưng", "n_features": "Số biến", "delta_std_mae": "PEMS08 ΔStd-MAE"}).merge(
    grp04_df[["group", "delta_std_mae", "delta_mae_speed"]].rename(columns={"group": "Nhóm đặc trưng", "delta_std_mae": "PEMS04 ΔStd-MAE", "delta_mae_speed": "PEMS04 ΔSpeed"}),
    on="Nhóm đặc trưng",
)
display(grp_cmp_df.round(5))
display(m_rank.sort_values("PEMS04 ΔStd-MAE", ascending=False).reset_index(drop=True).round(5))

attn04_res = extract_gatv2_edge_attention(
    model=model04_full,
    bundle=bundle04,
    feature_indices=fidx14,
    target_horizon_step=3,
    top_node=None,
    batch_size=256,
)
sm4 = attn04_res["summary_means"]
attn04_agg = pd.DataFrame(
    [
        {"Hướng cạnh GATv2 (PEMS04)": "Self (Bản thân)", "Bình thường": round(sm4["self_normal"], 4), "Tiền tắc nghẽn": round(sm4["self_pre_cong"], 4), "Dịch chuyển": round(sm4["self_pre_cong"] - sm4["self_normal"], 4)},
        {"Hướng cạnh GATv2 (PEMS04)": "In (Thượng nguồn)", "Bình thường": round(sm4["in_normal"], 4), "Tiền tắc nghẽn": round(sm4["in_pre_cong"], 4), "Dịch chuyển": round(sm4["in_pre_cong"] - sm4["in_normal"], 4)},
        {"Hướng cạnh GATv2 (PEMS04)": "Out (Hạ nguồn dội ngược)", "Bình thường": round(sm4["out_normal"], 4), "Tiền tắc nghẽn": round(sm4["out_pre_cong"], 4), "Dịch chuyển": round(sm4["out_pre_cong"] - sm4["out_normal"], 4)},
    ]
)
print(f"Phân bổ chú ý GATv2 trên {len(attn04_res['bn_summary_df'])} trạm điểm nghẽn của PEMS04 (Điểm nghẽn #1: N{attn04_res['top_node']}):")
display(attn04_agg)

plot_cross_map_pfi_and_attention(
    grp08_df,
    grp04_df,
    attn04_res,
    FIG_DIR / "nb05_cross_map_pfi_and_attention.png",
)

> **Nhận xét (Tính nhất quán cực cao của hệ đặc trưng và cơ chế chú ý `GATv2` giữa hai thành phố):**
>
> * **Thứ hạng đóng góp `4` nhóm đặc trưng lặp lại `100%` y nguyên giữa `PEMS08` và `PEMS04` (Hình 3A):** Trên cả hai mạng lưới ở hai khu vực địa lý độc lập, **toàn bộ `14 / 14` đặc trưng đều có `ΔStd-MAE > 0`** và giữ nguyên thứ tự: **Nhóm 4 `STID` (`+0,05263` trên `PEMS04` vs `+0,06865` trên `PEMS08`) > Nhóm 3 Độ dốc có dấu (`+0,02282` vs `+0,02478`, dẫn đầu là `occ_slope +0,02179` và `spd_slope +0,00594`) > Nhóm 1 Dao động khử dốc (`+0,00585` vs `+0,00514`, dẫn đầu là `occ_rstd +0,00303`) > Nhóm 2 Liên kết đa biến (`+0,00231` vs `+0,00423`, dẫn đầu là `flow_occ_decouple +0,00178`)**. Tương quan hạng Spearman của đủ `14` biến đạt **`ρ = 0,9956` ($p = 1,03 \times 10^{-13}$)** (`11` biến bổ sung đạt **`ρ = 0,9909`**, $p = 3,76 \times 10^{-9}$).
>
> * **Xu hướng dịch chuyển chú ý `GATv2` sang hạ nguồn (`Out`) còn mạnh hơn trên `PEMS04` (Hình 3B):** Trên trung bình `153` trạm điểm nghẽn của `PEMS04`, khi bước vào pha tiền tắc nghẽn, tổng trọng số chú ý dồn cho các nhánh lân cận tăng từ `60,9% → 65,9%` (`+5,0%pt`), trong đó chiều **`Out` (hạ nguồn dội ngược)** tăng mạnh nhất từ **`0,1817 → 0,2226` (`+0,0409`, tương đương `+22,5%` tương đối)** và chiều `In` tăng `0,4272 → 0,4365` (`+0,0093`). Tại điểm nghẽn #1 **`N79`**, tổng chú ý chiều `Out` tăng vọt từ `23,2% → 33,6%` (`+45,2%` tương đối) cùng `In` tăng `44,8% → 55,8%`.
>
> **Câu hỏi nghi vấn tiếp theo:**
>
> Khi đặt cạnh nhau trọn bộ kết quả đối chứng công bằng trên **cả `PEMS08` lẫn `PEMS04`**, bức tranh tổng thể qua từng tầm dự báo (`15p / 30p / 60p`) và đánh giá phụ đánh dấu tắc nghẽn cho thấy những kết luận chốt nào?

---
## 4. Tổng hợp Đối chiếu Hai Thành phố (`PEMS08` vs. `PEMS04`) & Độ nhạy Tầm Dự báo

Đặt cạnh nhau bảng tổng hợp đối chứng hoàn toàn công bằng trên **cùng kiến trúc `ST-GAT (LSTM → 2-layer GATv2)`** cho cả `PEMS08` (`NB04`) và `PEMS04` (`NB05`), cùng chi tiết `15p / 30p / 60p` và đánh giá phụ đánh dấu tắc nghẽn trên `PEMS04`.

In [ ]:
ablation08_df = pd.read_csv(OUT_DIR / "nb04_ablation_summary.csv")

# Master symmetric cross-map comparison table (PEMS08 vs PEMS04, same architecture, same 6 configs)
master_df = ablation08_df[
    ["Cấu hình", "Số kênh", "Std-MAE (3 biến)", "MAE Speed (mph)", "MAE Flow (xe/5p)", "So với Raw (%)", "So với Base (%)"]
].rename(
    columns={
        "Std-MAE (3 biến)": "PEMS08 Std-MAE",
        "MAE Speed (mph)": "PEMS08 Speed",
        "MAE Flow (xe/5p)": "PEMS08 Flow",
        "So với Raw (%)": "PEMS08 vs Raw",
        "So với Base (%)": "PEMS08 vs Base",
    }
).merge(
    ablation04_df[
        ["Cấu hình", "Std-MAE (3 biến)", "MAE Speed (mph)", "MAE Flow (xe/5p)", "So với Raw (%)", "So với Base (%)"]
    ].rename(
        columns={
            "Std-MAE (3 biến)": "PEMS04 Std-MAE",
            "MAE Speed (mph)": "PEMS04 Speed",
            "MAE Flow (xe/5p)": "PEMS04 Flow",
            "So với Raw (%)": "PEMS04 vs Raw",
            "So với Base (%)": "PEMS04 vs Base",
        }
    ),
    on="Cấu hình",
)
master_df.to_csv(OUT_DIR / "nb05_cross_map_master_summary.csv", index=False)
print("Bảng 4.1 — Tổng hợp đối chứng đối xứng 6 cấu hình trên cả 2 thành phố (cùng kiến trúc ST-GAT LSTM → GATv2):")
display(master_df)

# PEMS04 detail by horizon (15p / 30p / 60p) for Raw(3), Base(7), Full(14)
h04_rows = []
for h_m in [15, 30, 60]:
    for cfg in ["Raw (3)", "Base (7)", "Full (14)"]:
        hm = results04[cfg]["test_eval"]["by_horizon"][h_m]
        h04_rows.append(
            {
                "Tầm dự báo": f"{h_m} phút",
                "Cấu hình (PEMS04)": cfg,
                "Std-MAE": round(hm["std_mae"], 4),
                "MAE Speed (mph)": round(hm["mae_speed"], 3),
                "MAE Occ": round(hm["mae_occ"], 5),
                "MAE Flow (xe/5p)": round(hm["mae_flow"], 2),
                "MAE Speed Điểm nghẽn": round(hm["mae_speed_bn"], 3),
                "F1 Tắc (phụ)": round(hm["cong_f1"], 4),
                "Precision (phụ)": round(hm["cong_prec"], 4),
                "Recall (phụ)": round(hm["cong_rec"], 4),
            }
        )
h04_df = pd.DataFrame(h04_rows)
h04_df.to_csv(OUT_DIR / "nb05_pems04_horizon_results.csv", index=False)
print("Bảng 4.2 — Chi tiết PEMS04 tại 15p / 30p / 60p (MAE chính & Đánh giá phụ đánh dấu tắc nghẽn):")
display(h04_df)

# 12-step MAE curves for Figure 4
horizon_curves04 = {}
for cname, ev_obj in [
    ("Zero-Shot Full (14)", zs_full_eval),
    ("Raw (3)", results04["Raw (3)"]["test_eval"]),
    ("Base (7)", results04["Base (7)"]["test_eval"]),
    ("Full (14)", results04["Full (14)"]["test_eval"]),
]:
    ae = np.abs(ev_obj["y_pred_raw"] - ev_obj["y_true_raw"])
    horizon_curves04[cname] = {"mae_speed": ae[..., 2].mean(axis=(0, 1))}

cong04_df = pd.DataFrame(
    [
        {
            "Tầm dự báo": f"{h} phút",
            "F1 Raw (3)": round(results04["Raw (3)"]["test_eval"]["by_horizon"][h]["cong_f1"], 4),
            "F1 Base (7)": round(results04["Base (7)"]["test_eval"]["by_horizon"][h]["cong_f1"], 4),
            "F1 Full (14)": round(results04["Full (14)"]["test_eval"]["by_horizon"][h]["cong_f1"], 4),
        }
        for h in [15, 30, 60]
    ]
)

ov_r4 = results04["Raw (3)"]["test_eval"]["overall"]
ov_b4 = results04["Base (7)"]["test_eval"]["overall"]
ov_f4 = results04["Full (14)"]["test_eval"]["overall"]
print(
    f"Đánh giá phụ PEMS04 (Avg 5–60p): "
    f"Precision: Raw={ov_r4['cong_prec']:.4f} -> Base={ov_b4['cong_prec']:.4f} -> Full={ov_f4['cong_prec']:.4f} | "
    f"F1: Raw={ov_r4['cong_f1']:.4f} -> Base={ov_b4['cong_f1']:.4f} -> Full={ov_f4['cong_f1']:.4f}"
)

plot_final_cross_map_synthesis(
    ablation08_df,
    ablation04_df,
    horizon_curves04,
    cong04_df,
    FIG_DIR / "nb05_final_cross_map_synthesis.png",
)

> **Nhận xét (Tổng hợp đối xứng `PEMS08` vs. `PEMS04` — Công bằng tuyệt đối trên cùng kiến trúc `ST-GAT`):**
>
> * **Bức tranh nhất quán trên cả hai mạng lưới (Bảng 4.1 & Hình 4A–B):** Bổ sung đủ `4` nhóm đặc trưng (`Full 14`) vào cùng kiến trúc `ST-GAT (LSTM → GATv2)` gốc giảm mạnh `Test Std-MAE` trên **cả hai bản đồ độc lập**: **`-8,19%`** trên `PEMS08` (`0,1771 → 0,1626`, `-2,17%` so với `Base`) và **`-6,25%`** trên `PEMS04` (`0,1904 → 0,1785`, `-1,38%` so với `Base`). Trên `PEMS04`, khoảng cách cải thiện cũng mở rộng rõ nhất ở tầm xa **`60 phút`** (`MAE Speed: 2,387 → 2,261 → 2,225 mph`, giảm `-6,79%` vs `Raw`; `MAE Flow: 30,81 → 26,75 → 26,43`, giảm **`-14,22%`** vs `Raw`).
>
> * **Đánh giá phụ đánh dấu tắc nghẽn trên `PEMS04` (Bảng 4.2 & Hình 4C — ghi nhận trung thực):** `Full (14)` đạt **`Precision` cao nhất trên mọi mốc thời gian** (`Avg: 0,7862 → 0,8060 → 0,8308`, tăng **`+4,46%pt`** so với `Raw`; mốc `60p` tăng từ `0,6696 → 0,7588`, **`+8,92%pt`**) nhờ giảm mạnh báo động giả. Tuy nhiên, do huấn luyện bằng hàm mất mát hồi quy `L1` đều toàn mạng (không gán trọng số riêng cho `3%` đuôi tắc sâu), độ thận trọng cao khiến `Recall` của `Full (14)` (`0,6180`) thấp hơn nhẹ so với `Base (7)` (`0,6492`), nên `F1` của `Full (14)` (`0,7088`, `60p: 0,5434`) cao hơn `Raw (3)` (`0,7065`, `60p: 0,5290`) nhưng thấp hơn nhẹ so với `Base (7)` (`0,7192`, `60p: 0,5668`) — một giới hạn khách quan được ghi nhận rõ trong Phần 5.

---
## 5. Kết luận Toàn bộ Dự án (`RQ1 + RQ2`), Đóng góp Khoa học và Giới hạn Nghiên cứu

Qua lộ trình `5` notebook khép kín trên hai mạng lưới cao tốc thực tế Caltrans PeMS (**`PEMS08`** `170` trạm làm tập nghiên cứu chính và **`PEMS04`** `307` trạm làm tập kiểm chứng độc lập hoàn toàn), dự án đã trả lời trọn vẹn cả hai câu hỏi nghiên cứu đề ra với kỷ luật chống rò rỉ dữ liệu và đối chứng công bằng *ceteris paribus* tuyệt đối:

### 5.1. Tổng kết Đóng góp Chính

1. **Tầng Tiền xử lý Chuẩn hóa (`NB01 – NB02`):** Áp dụng thống nhất thuật toán chẩn đoán `4` chỉ số ngày ($S_1 \dots S_4$) và hồi quy tuyến tính cặp lân cận của **Chen et al. (*TRR*, 2003)** giúp tách bạch đường vắng ban đêm hợp lệ khỏi lỗi mất xung `0.0` và lỗi đóng băng dương ($S_4$), đưa cả `PEMS08` lẫn `PEMS04` về **`0` lỗi tồn đọng** mà vẫn bảo toàn phân phối và dao động 5 phút.

2. **Trả lời `RQ1` — Xác định Tắc nghẽn Hướng Dữ liệu & Hệ `14` Đặc trưng Tiền tắc nghẽn (`NB03` trên `PEMS08`, kiểm chứng trên `PEMS04` ở `NB05`):**
   * Công thức tự hiệu chuẩn ngưỡng từ Biểu đồ cơ bản kết hợp quy chuẩn **FHWA (`< 60%` FFS)** tự động thích nghi xuyên bản đồ thuần trên `Train (60%)` mà không chép cứng hằng số: $v_{\text{cong}} = 40{,}26\text{ mph}$, $o_{\text{sat}} = 0{,}1375$ (`PEMS08`) và tự hạ xuống $v_{\text{cong}} = 40{,}32\text{ mph}$, $o_{\text{sat}} = 0{,}1025$ (`PEMS04`), đồng thời chứng minh loại trừ ngưỡng `Flow` do *Capacity Drop* (`-13,7%`).
   * Xây dựng trọn bộ **`14` đặc trưng cục bộ nhân quả (`H = 12` = `60 phút`)**: `3` biến gốc + `4` chu kỳ `STID` + `7` chỉ báo tiền tắc nghẽn (`3` Dao động khử dốc + `2` Liên kết đa biến CSD + `2` Độ dốc có dấu), nâng `Val PR-AUC` tuyến tính thêm **`+22,51%` đến `+31,66%`** trên `PEMS08`.
   * Bác bỏ hình học đồ thị tĩnh (`Betweenness`: `p = 0,3966`) và chứng minh tương tác động có hướng 2 chiều `In + Out` nâng tiếp `Val PR-AUC` thêm `+1,88%` (`5p`) đến `+5,48%` (`30p`), nhất quán với cơ chế độ trễ lan truyền (**PDFormer, 2023**).

3. **Trả lời `RQ2` — Đóng góp của Các Nhóm Đặc trưng trên `ST-GAT (LSTM → 2-layer GATv2)` (`NB04 – NB05`):**
   * **Đối chứng hoàn toàn công bằng trên cùng kiến trúc đóng băng (`11.524 – 12.932` tham số, `L1Loss / MAE`, `SEED = 42`):** Bổ sung đủ `4` nhóm đặc trưng (`Full 14`) giảm mạnh `Test Std-MAE` trên **cả hai bản đồ độc lập**: **`-8,19%`** so với `Raw (3)` (`-2,17%` so với `Base 7`) trên **`PEMS08`** (`0,1771 → 0,1662 → 0,1626`) và **`-6,25%`** so với `Raw (3)` (`-1,38%` so với `Base 7`) trên **`PEMS04`** (`0,1904 → 0,1810 → 0,1785`).
   * **Nhất quán cả ở cấp độ từng nhóm đặc trưng và từng biến:** Trên cả hai thành phố, **Nhóm 4 `STID`** tạo bước nhảy đơn nhóm lớn nhất trong mô hình phi tuyến (`-6,15%` trên `PEMS08`, `-4,94%` trên `PEMS04`), **Nhóm 1 Dao động khử dốc** đứng #1 trong `3` nhóm tín hiệu tiền tắc chạy riêng lẻ, và **`Full (14)`** dẫn đầu tuyệt đối trên mọi biến `Speed / Occ / Flow` và mọi tầm dự báo (`15p / 30p / 60p`, mạnh nhất ở **`60 phút`**).
   * **Độ lặp lại cấu trúc giải thích xuyên hai thành phố:** Trong phép đo `Test PFI` trên `Full (14)`, toàn bộ **`14 / 14` đặc trưng** đều có đóng góp dương (`ΔStd-MAE > 0`) trên cả `PEMS08` và `PEMS04` với tương quan hạng **Spearman `ρ = 0,9956` ($p = 1,03 \times 10^{-13}$)**, giữ nguyên thứ tự `4` nhóm (**`STID` > `Độ dốc có dấu (occ_slope / spd_slope)` > `Dao động khử dốc (occ_rstd)` > `Liên kết đa biến (flow_occ_decouple)`**). Trong chuyển giao *Zero-Shot* (`PEMS08 → PEMS04`, `0` epoch), `7` tín hiệu tiền tắc nghẽn vật lý tỏ ra bền vững hơn `STID` trước dịch chuyển miền địa lý; đồng thời trọng số chú ý `GATv2` trên cả hai bản đồ đều tự động dồn mạnh sang chiều hạ nguồn dội ngược (`Out`) trước khi tắc nghẽn bùng phát (`+66,7%` tại `N22` trên `PEMS08` và `+22,5%` trên `153` điểm nghẽn `PEMS04`).

### 5.2. Giới hạn Nghiên cứu và Hướng Phát triển

1. **Tối ưu hóa hồi quy thuần `MAE` cho sự kiện hiếm (`2%–3%`):** Hàm mất mát `L1` cân bằng cho toàn mạng giúp `Full (14)` giảm mạnh `MAE` liên tục và đạt `Precision` đánh dấu tắc rất cao (`0,82–0,83`), nhưng đánh đổi nhẹ `Recall` ở đuôi tắc cực đoan (`60p` khoảng `0,42–0,44`). Hướng mở rộng tự nhiên là kết hợp bộ `14` đặc trưng với hàm mất mát có trọng số theo pha tắc nghẽn (*congestion-weighted loss*).
2. **Dịch chuyển miền của đặc trưng chu kỳ thời gian (`STID`) khi chuyển giao *Zero-Shot*:** Khi triển khai *Zero-Shot* không huấn luyện lại sang thành phố khác mùa và khác khung giờ cao điểm (`PEMS08 → PEMS04`), các chỉ báo vật lý bất biến (`Nhóm 1–3`) chuyển giao tốt hơn `STID`; việc tinh chỉnh nhẹ (*few-shot fine-tuning*) trên vài ngày đầu của bản đồ mới có thể khắc phục nhanh độ lệch khung giờ này.
3. **Phạm vi dữ liệu và đánh giá đa kiến trúc mở rộng:** Hai tập `PEMS08` (`62` ngày) và `PEMS04` (`59` ngày) chưa bao phủ đủ biến động theo mùa cả năm hay sự cố tai nạn phi chu kỳ. Trong nghiên cứu tiếp theo, bộ `7` chỉ báo tiền tắc nghẽn minh bạch của đề tài có thể được thử nghiệm tích hợp làm đầu vào cho các kiến trúc STGNN / Transformer quy mô lớn hơn (như trên bộ dữ liệu dài hạn `LargeST`).

---
**Hoàn tất trọn vẹn toàn bộ lộ trình `5` Notebook (`NB01 – NB05`) — Sẵn sàng biên soạn bài báo chuẩn IEEE ($\le 3.000$ từ) và bài thuyết trình 10 phút.**